Install transformers and datasets

In [1]:
! pip install datasets transformers seqeval


# Fine-tuning a model on a token classification task

In [2]:
task = "ner"
batch_size = 16

## Loading the dataset

We will use the Datasets library to download the data and get the metric we need to use for evaluation (to compare our model to the benchmark). This can be easily done with the functions `load_dataset` and `load_metric`.  

In [3]:
!pip install evaluate

In [4]:
from datasets import load_dataset,Dataset

In [5]:
# Then import like this:
import evaluate

The code below allow us to upload and read the data from Google Drive

In [6]:
from google.colab import drive
drive.mount('/content/gdrive')

Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).


In [7]:
import os
os.environ['KAGGLE_CONFIG_DIR']='/content/gdrive/My Drive/ABSAA/SemEval2016_arabic'
#os.environ['KAGGLE_CONFIG_DIR']='/content/gdrive/My Drive/educationdata'

In [8]:
ls

gdrive/  sample_data/


In [9]:
%cd /content/gdrive/My Drive/ABSAA//educationdata
#%cd gdrive/My Drive/educationdata

/content/gdrive/My Drive/ABSAA/educationdata


In [10]:
ls


bert-finetuned-sem_eval-english/  reviewsAnnotator3.xml  test-ner/
dataset_polarities.json           reviewsannotator4.xml  user3_reviews.xml
reviewsAnnotator1.xml             reviewsUser4.xml       wandb/
reviewsAnnotator2.xml             testData.xml


In [11]:
import re

def clean(text):
    # Replace <br/> tags with space
    text = text.replace("<br/>", " ")

    # Keep only Arabic characters, Arabic diacritics, and Arabic punctuation
    # This includes:
    # - Arabic letters (\u0621-\u064A)
    # - Arabic diacritics (\u064B-\u065F)
    # - Arabic Extended letters (\u0671-\u06D3)
    # - Arabic punctuation (\u060C, \u061B, \u061F, \u0660-\u0669 for Arabic numbers)
    # - Space to separate words
    arabic_pattern = re.compile(
        r'[^\u0600-\u06FF\u0750-\u077F\u08A0-\u08FF\uFB50-\uFDFF\uFE70-\uFEFF ]'
    )

    # Remove all non-Arabic characters
    text = re.sub(arabic_pattern, " ", text)

    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()

    return text

In [12]:
import xml.etree.ElementTree as ET
import pandas as pd
import re
from typing import List, Tuple, Dict, Set

def tokenize_with_positions(text: str) -> List[Tuple[str, Tuple[int, int]]]:
    """
    Tokenize text and return tokens with their character positions.
    """
    tokens = []
    positions = []

    # Utilisation d'une regex pour capturer les mots et la ponctuation
    pattern = r'\w+|[^\w\s]'
    for match in re.finditer(pattern, clean(text)):
        token = match.group()
        start, end = match.span()
        tokens.append(token)
        positions.append((start, end))

    return tokens, positions

def polarity_to_num(polarity: str) -> int:
    """
    Convert polarity string to numeric code.
    """
    pol = polarity.lower()
    if pol == 'positive':
        return 1
    elif pol == 'negative':
        return 2
    elif pol == 'neutral':
        return 3
    elif pol == 'conflict':
        return 4
    else:
        return 0  # Par défaut

def find_token_indices_for_aspect(positions: List[Tuple[int, int]],
                                  aspect_start: int,
                                  aspect_end: int) -> List[int]:
    """
    Trouve tous les indices de tokens qui font partie de l'aspect,
    même s'ils ne sont pas entièrement contenus dans le span.
    """
    aspect_token_indices = []

    for i, (token_start, token_end) in enumerate(positions):

        if (token_start >= aspect_start and token_start < aspect_end) or \
           (token_end > aspect_start and token_end <= aspect_end) or \
           (token_start <= aspect_start and token_end >= aspect_end):
            aspect_token_indices.append(i)


    aspect_token_indices = sorted(set(aspect_token_indices))


    if aspect_token_indices:
        continuous_indices = [aspect_token_indices[0]]
        for idx in aspect_token_indices[1:]:
            if idx == continuous_indices[-1] + 1:
                continuous_indices.append(idx)
        return continuous_indices

    return aspect_token_indices

def process_sentence(sentence_elem) -> dict:

    text_elem = sentence_elem.find('text')
    if text_elem is None or text_elem.text is None:
        return None

    text = text_elem.text.strip()


    tokens, positions = tokenize_with_positions(clean(text))


    ner_tags = ['O'] * len(tokens)
    ner_tags1 = [0] * len(tokens)


    aspect_terms = sentence_elem.findall('.//aspectTerm')


    marked_tokens = set()


    for aspect in aspect_terms:
        polarity = aspect.get('polarity', '').lower()


        if not polarity:
            continue

        try:
            start = int(aspect.get('from', 0))
            end = int(aspect.get('to', 0))
        except ValueError:
            continue


        aspect_token_indices = find_token_indices_for_aspect(positions, start, end)


        available_indices = [idx for idx in aspect_token_indices if idx not in marked_tokens]

        if not available_indices:
            continue


        first_idx = available_indices[0]
        ner_tags[first_idx] = f'B-{polarity}'
        ner_tags1[first_idx] = polarity_to_num(polarity)
        marked_tokens.add(first_idx)


        for idx in available_indices[1:]:
            ner_tags[idx] = f'I-{polarity}'
            ner_tags1[idx] = polarity_to_num(polarity)
            marked_tokens.add(idx)

    return {
        'sentence': (clean(text)),
        'tokens': tokens,
        'ner_tags': ner_tags,
        'ner_tags1': ner_tags1
    }

def xml_to_dataframe(xml_file_path: str) -> pd.DataFrame:


    tree = ET.parse(xml_file_path)
    root = tree.getroot()


    data = []
    for sentence_elem in root.findall('sentence'):
        result = process_sentence(sentence_elem)
        if result:
            data.append(result)


    df = pd.DataFrame(data)
    return df

# Fonction utilitaire pour afficher les résultats de façon lisible
def print_sentence_analysis(df, sentence_index=0):
    """
    Affiche une analyse détaillée d'une phrase.
    """
    if sentence_index >= len(df):
        print(f"Index {sentence_index} hors limites. Le DataFrame a {len(df)} phrases.")
        return

    example = df.iloc[sentence_index]
    print("=" * 80)
    print(f"Phrase {sentence_index}:")
    print(f"Texte complet: {example['sentence']}")
    print("-" * 80)
    print("Analyse token par token:")
    print(f"{'Token':<20} {'Position':<15} {'NER Tag':<15} {'NER Tag1':<10}")
    print("-" * 80)

    for i, (token, ner_tag, ner_tag1) in enumerate(zip(example['tokens'], example['ner_tags'], example['ner_tags1'])):
        print(f"{token:<20} {str(i):<15} {ner_tag:<15} {ner_tag1:<10}")

    # Afficher les aspects détectés
    print("-" * 80)
    print("Aspects détectés:")
    current_aspect = None
    aspect_tokens = []

    for i, (token, ner_tag) in enumerate(zip(example['tokens'], example['ner_tags'])):
        if ner_tag.startswith('B-'):
            if current_aspect:
                print(f"  - {current_aspect}: {' '.join(aspect_tokens)}")
            current_aspect = ner_tag[2:]
            aspect_tokens = [token]
        elif ner_tag.startswith('I-') and current_aspect:
            aspect_tokens.append(token)
        elif current_aspect and not ner_tag.startswith('I-'):
            print(f"  - {current_aspect}: {' '.join(aspect_tokens)}")
            current_aspect = None
            aspect_tokens = []

    if current_aspect:
        print(f"  - {current_aspect}: {' '.join(aspect_tokens)}")

    print("=" * 80)

# Exemple d'utilisation
if __name__ == "__main__":
    import pandas as pd
    df = pd.DataFrame()
    # Convertir le fichier XML
    # Récupérer tous les fichiers XML du dossier
    fichiers = [f for f in os.listdir() if f.endswith('.xml')]
    for fichier in fichiers:
       dff = xml_to_dataframe(fichier)
       df = pd.concat([df, dff], ignore_index=True)

    # Afficher les premières lignes
    print("DataFrame créé avec succès!")
    print(f"Nombre de phrases: {len(df)}")
    print("\nPremières lignes du DataFrame:")
    print(df.head())

    # Afficher un exemple détaillé d'une phrase
    if len(df) > 0:
        print_sentence_analysis(df, 0)

        # Afficher quelques autres phrases pour vérification
        print("\n\nVérification rapide d'autres phrases:")
        for i in range(min(3, len(df))):
            example = df.iloc[i]
            # Compter les tokens marqués
            marked_count = sum(1 for tag in example['ner_tags'] if tag != 'O')
            print(f"Phrase {i}: {len(example['tokens'])} tokens, {marked_count} tokens marqués")
    ''' # Sauvegarder en CSV si nécessaire
    df.to_csv('output_dataframe.csv', index=False, encoding='utf-8')
    print("\nDataFrame sauvegardé dans 'output_dataframe.csv'")'''


DataFrame créé avec succès!
Nombre de phrases: 1360

Premières lignes du DataFrame:
                                            sentence  \
0  التعليم في المغرب إلى أين وزارة اللا تربية وال...   
1  كان الله في عون التعليم بالمغرب ابمثل هؤلاء ال...   
2  يمكنكم ارتداء القفطان والادعاء كما تشاؤون، لكن...   
3  رحمة الله على التعليم في المغرب قمة التخربيق و...   
4  تعيين الفنانة لطيفة أحرار كعضو في مجلس الوكالة...   

                                              tokens  \
0  [التعليم, في, المغرب, إلى, أين, وزارة, اللا, ت...   
1  [كان, الله, في, عون, التعليم, بالمغرب, ابمثل, ...   
2  [يمكنكم, ارتداء, القفطان, والادعاء, كما, تشاؤو...   
3  [رحمة, الله, على, التعليم, في, المغرب, قمة, ال...   
4  [تعيين, الفنانة, لطيفة, أحرار, كعضو, في, مجلس,...   

                                            ner_tags  \
0  [B-negative, I-negative, I-negative, O, O, B-n...   
1  [O, O, O, O, B-negative, I-negative, O, O, B-n...   
2  [O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ...   
3  [O, O, O, B-neg

In [13]:
example

,2
sentence,يمكنكم ارتداء القفطان والادعاء كما تشاؤون، لكن...
tokens,"[يمكنكم, ارتداء, القفطان, والادعاء, كما, تشاؤو..."
ner_tags,"[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ..."
ner_tags1,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."


In [14]:
def dataframe_to_dict_multiple(df):
    """
    Convertit un DataFrame avec plusieurs phrases en liste de dictionnaires
    """
    result = []

    for _, row in df.iterrows():
        # Conversion des colonnes en listes si nécessaire
        tokens = row['tokens']
        if isinstance(tokens, str):
            try:
                tokens = ast.literal_eval(tokens)
            except:
                tokens = [tokens]

        ner_tags = row['ner_tags']
        if isinstance(ner_tags, str):
            try:
                ner_tags = ast.literal_eval(ner_tags)
            except:
                ner_tags = [ner_tags]

        ner_tags1 = row['ner_tags1']
        if isinstance(ner_tags1, str):
            try:
                ner_tags1 = ast.literal_eval(ner_tags1)
            except:
                ner_tags1 = [ner_tags1]

        result.append({
            'sentence': row['sentence'],
            'tokens': tokens,
            'ner_tags': ner_tags,
            'ner_tags1': ner_tags1
        })

    return result
print("\nAvec plusieurs phrases:")
multiple_dicts = dataframe_to_dict_multiple(df)
for i, d in enumerate(multiple_dicts):
    print(f"\nPhrase {i+1}: {d}")


Avec plusieurs phrases:

Phrase 1: {'sentence': 'التعليم في المغرب إلى أين وزارة اللا تربية والتهديم إيديسيون تقدم', 'tokens': ['التعليم', 'في', 'المغرب', 'إلى', 'أين', 'وزارة', 'اللا', 'تربية', 'والتهديم', 'إيديسيون', 'تقدم'], 'ner_tags': ['B-negative', 'I-negative', 'I-negative', 'O', 'O', 'B-negative', 'I-negative', 'I-negative', 'I-negative', 'I-negative', 'O'], 'ner_tags1': [2, 2, 2, 0, 0, 2, 2, 2, 2, 2, 0]}

Phrase 2: {'sentence': 'كان الله في عون التعليم بالمغرب ابمثل هؤلاء الوزراء سينهض المغرب بالتعليم؟ سحقا للزبونية والتحزب في رد على سؤال للنائبة البرلمانية فاطمة التامني ظهر وزير التربية الوطنية برادة متلعتما ومرتبكا لا يستطيع الجواب عن السؤال المطروح واستعان بورقة مكتوبة لا تسمن ولا تغني من جوع', 'tokens': ['كان', 'الله', 'في', 'عون', 'التعليم', 'بالمغرب', 'ابمثل', 'هؤلاء', 'الوزراء', 'سينهض', 'المغرب', 'بالتعليم', '؟', 'سحقا', 'للزبونية', 'والتحزب', 'في', 'رد', 'على', 'سؤال', 'للنائبة', 'البرلمانية', 'فاطمة', 'التامني', 'ظهر', 'وزير', 'التربية', 'الوطنية', 'برادة', 'متلعتما

In [15]:
multiple_dicts[0]

{'sentence': 'التعليم في المغرب إلى أين وزارة اللا تربية والتهديم إيديسيون تقدم',
 'tokens': ['التعليم',
  'في',
  'المغرب',
  'إلى',
  'أين',
  'وزارة',
  'اللا',
  'تربية',
  'والتهديم',
  'إيديسيون',
  'تقدم'],
 'ner_tags': ['B-negative',
  'I-negative',
  'I-negative',
  'O',
  'O',
  'B-negative',
  'I-negative',
  'I-negative',
  'I-negative',
  'I-negative',
  'O'],
 'ner_tags1': [2, 2, 2, 0, 0, 2, 2, 2, 2, 2, 0]}

In [16]:
#multiple_dicts1 = dataframe_to_dict_multiple(df1)

in this corpus, ten types of clinical entities were annotated: Anatomy, Chemical and Drugs, Devices, Disorders, Geographic Areas, Living Beings, Objects, Phenomena, Physiology, Procedures
with the labels: ANAT, CHEM, DEVI, DISO, GEOG, LIVB, OBJC, PHEN, PHYS, PROC.

The function tagtoID() convert every tag to an adequat ID

In [17]:

from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np

# Mélanger le DataFrame d'abord (important pour éviter les biais)
#df = df.sample(frac=1, random_state=42).reset_index(drop=True)

# 1. Séparer d'abord train+val (80%) et test (20%)
train_val_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    shuffle=True
)



Count the number of every tag in train data

In [18]:
# Note: 10%/80% = 0.125 pour obtenir 10% du total original
train_df, val_df = train_test_split(
    train_val_df,
    test_size=0.125,  # 10% / 80% = 0.125
    random_state=42,
    shuffle=True
)

In [19]:
multiple_dicts_train = dataframe_to_dict_multiple(train_df)
multiple_dicts_test = dataframe_to_dict_multiple(test_df)
multiple_dicts_val = dataframe_to_dict_multiple(val_df)

Count the number of every tag in test data

In the test data(MEDLINE data) by going through each file and we recover each word with the appropriate label

Convert the data to dataset

In [20]:

import pandas as pd
# df = pd.DataFrame({"a": [1, 2, 3]})
dataset1 = Dataset.from_pandas(pd.DataFrame.from_dict(multiple_dicts_train))
dataset2 = Dataset.from_pandas(pd.DataFrame.from_dict(multiple_dicts_test))
dataset3 = Dataset.from_pandas(pd.DataFrame.from_dict(multiple_dicts_val))




In [21]:
dataset3

Dataset({
    features: ['sentence', 'tokens', 'ner_tags', 'ner_tags1'],
    num_rows: 136
})

Put train data, test data,validation data in the same datasets

In [22]:
from datasets  import DatasetDict
datasets=DatasetDict({'train': dataset1,'validation': dataset3,'test': dataset2})

In [23]:
datasets

DatasetDict({
    train: Dataset({
        features: ['sentence', 'tokens', 'ner_tags', 'ner_tags1'],
        num_rows: 952
    })
    validation: Dataset({
        features: ['sentence', 'tokens', 'ner_tags', 'ner_tags1'],
        num_rows: 136
    })
    test: Dataset({
        features: ['sentence', 'tokens', 'ner_tags', 'ner_tags1'],
        num_rows: 272
    })
})

. The notebook should work with any token classification dataset provided by the Datasets library. If you're using your own dataset defined from a JSON or csv file

The `datasets` object itself is DatasetDict, which contains one key for the training, validation and test set.

In [24]:
#datasets1

We can see the training, validation and test sets all have a column for the tokens (the input texts split into words) and one column of labels for each kind of task we introduced before.

To access an actual element, you need to select a split first, then give an index:

In [25]:
datasets["train"][0]

{'sentence': 'لا يزال التعليم في المغرب، كما هو معلوم، يتخبط لإيجاد خلطة سحرية للخروج من عنق الزجاجة، وذلك منذ مدة ليست بالقصيرة والنتيجة دائما واحدة، وهي أننا لازلنا في ذيل الأمم التي تنبغ في التعليم وتؤتيه حقه من العناية والتقدير، لترقى به المقال المغرب فكر',
 'tokens': ['لا',
  'يزال',
  'التعليم',
  'في',
  'المغرب',
  '،',
  'كما',
  'هو',
  'معلوم',
  '،',
  'يتخبط',
  'لإيجاد',
  'خلطة',
  'سحرية',
  'للخروج',
  'من',
  'عنق',
  'الزجاجة',
  '،',
  'وذلك',
  'منذ',
  'مدة',
  'ليست',
  'بالقصيرة',
  'والنتيجة',
  'دائما',
  'واحدة',
  '،',
  'وهي',
  'أننا',
  'لازلنا',
  'في',
  'ذيل',
  'الأمم',
  'التي',
  'تنبغ',
  'في',
  'التعليم',
  'وتؤتيه',
  'حقه',
  'من',
  'العناية',
  'والتقدير',
  '،',
  'لترقى',
  'به',
  'المقال',
  'المغرب',
  'فكر'],
 'ner_tags': ['O',
  'O',
  'B-negative',
  'I-negative',
  'I-negative',
  'I-negative',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'B-neutral',
  'I-neutral',
  'I-neutral',
  'O',
  'O',
  'O',
  'O'

In [26]:
datasets["train"][1]

{'sentence': 'تهدف المذكرة إلى دعم المناهج الدراسية، وتنظيم ورش العمل، وبناء برامج تدريبية حديثة تخدم الطلبة وأعضاء الهيئة التدريسية في الجامعات العربيةفي خطوة جديدة نحو مستقبل التعليم القانوني الرقمي في العالم العربي و توجه استراتيجي نحو ربط التعليم القانوني بسوق العمل الحديث',
 'tokens': ['تهدف',
  'المذكرة',
  'إلى',
  'دعم',
  'المناهج',
  'الدراسية',
  '،',
  'وتنظيم',
  'ورش',
  'العمل',
  '،',
  'وبناء',
  'برامج',
  'تدريبية',
  'حديثة',
  'تخدم',
  'الطلبة',
  'وأعضاء',
  'الهيئة',
  'التدريسية',
  'في',
  'الجامعات',
  'العربيةفي',
  'خطوة',
  'جديدة',
  'نحو',
  'مستقبل',
  'التعليم',
  'القانوني',
  'الرقمي',
  'في',
  'العالم',
  'العربي',
  'و',
  'توجه',
  'استراتيجي',
  'نحو',
  'ربط',
  'التعليم',
  'القانوني',
  'بسوق',
  'العمل',
  'الحديث'],
 'ner_tags': ['O',
  'O',
  'O',
  'B-positive',
  'I-positive',
  'I-positive',
  'O',
  'B-positive',
  'I-positive',
  'I-positive',
  'O',
  'B-positive',
  'I-positive',
  'I-positive',
  'I-positive',
  'B-positive',
  'I-

The labels are already coded as integer ids to be easily usable by our model, but the correspondence with the actual categories is stored in the `features` of the dataset:

in this corpus, ten types of clinical entities were annotated: Anatomy, Chemical and Drugs, Devices, Disorders, Geographic Areas, Living Beings, Objects, Phenomena, Physiology, Procedures with the labels: ANAT, CHEM, DEVI, DISO, GEOG, LIVB, OBJC, PHEN, PHYS, PROC.

Since the labels are lists of `ClassLabel`, the actual names of the labels are nested in the `feature` attribute of the object above:

In [27]:
label_list = ['B-positive','O','B-negative','B-conflict','B-neutral']


To get a sense of what the data looks like, the following function will show some examples picked randomly in the dataset (automatically decoding the labels in passing).

In [28]:
from datasets import ClassLabel, Sequence
import random
import pandas as pd
from IPython.display import display, HTML

def show_random_elements(dataset, num_examples=10):
    assert num_examples <= len(dataset), "Can't pick more elements than there are in the dataset."
    picks = []
    for _ in range(num_examples):
        pick = random.randint(0, len(dataset)-1)
        while pick in picks:
            pick = random.randint(0, len(dataset)-1)
        picks.append(pick)

    df = pd.DataFrame(dataset[picks])
    for column, typ in dataset.features.items():
        if isinstance(typ, ClassLabel):
            df[column] = df[column].transform(lambda i: typ.names[i])
        elif isinstance(typ, Sequence) and isinstance(typ.feature, ClassLabel):
            df[column] = df[column].transform(lambda x: [typ.feature.names[i] for i in x])
    display(HTML(df.to_html()))

In [29]:
show_random_elements(datasets["train"])

,sentence,tokens,ner_tags,ner_tags1
0,في واقع التعليم العربي المعاصر، يواجه المعلم تحدياً في جعل المنهج الدراسي أكثر ارتباطاً بحياة الطالب واهتماماته فكثير من المناهج اليوم تُقدَّم بصورة جافة؛ حيث تفتقر إلى البُعد الإنساني والمعنوي؛ مما يقلل من فاعليتها في التأثير على سلوك المتعلم أو توسيع مداركه لذلك، فإن تطوير,"[في, واقع, التعليم, العربي, المعاصر, ،, يواجه, المعلم, تحديا, ً, في, جعل, المنهج, الدراسي, أكثر, ارتباطا, ً, بحياة, الطالب, واهتماماته, فكثير, من, المناهج, اليوم, ت, ُ, قد, َ, ّ, م, بصورة, جافة, ؛, حيث, تفتقر, إلى, الب, ُ, عد, الإنساني, والمعنوي, ؛, مما, يقلل, من, فاعليتها, في, التأثير, على, سلوك, المتعلم, أو, توسيع, مداركه, لذلك, ،, فإن, تطوير]","[O, O, O, O, O, O, O, O, O, O, O, O, B-negative, I-negative, O, B-negative, I-negative, I-negative, I-negative, I-negative, I-negative, O, B-negative, I-negative, I-negative, I-negative, I-negative, I-negative, I-negative, I-negative, I-negative, I-negative, I-negative, I-negative, O, O, O, O, O, O, O, O, O, O, B-negative, I-negative, I-negative, I-negative, I-negative, I-negative, I-negative, I-negative, I-negative, I-negative, I-negative, O, O, O]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 2, 0, 2, 2, 2, 2, 2, 2, 0, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 0, 0, 0]"
1,وأما المعلومة الحلوة عن الكتب والقراء فإليك ما أؤمن به كل الكتب جيدة، حتى الردئ منها، لأنه يحسن ذائقتك ويصقل عين الناقد وحسّه في داخلك لا تكمل كتاباً لم يعجبك الترجمة قد تظلم المحتوى، لذا ابحث عن الترجمات الموثوقة اقرأ الكتاب الكترونياً قبل أن تشتريه لا بأس,"[وأما, المعلومة, الحلوة, عن, الكتب, والقراء, فإليك, ما, أؤمن, به, كل, الكتب, جيدة, ،, حتى, الردئ, منها, ،, لأنه, يحسن, ذائقتك, ويصقل, عين, الناقد, وحس, ّ, ه, في, داخلك, لا, تكمل, كتابا, ً, لم, يعجبك, الترجمة, قد, تظلم, المحتوى, ،, لذا, ابحث, عن, الترجمات, الموثوقة, اقرأ, الكتاب, الكترونيا, ً, قبل, أن, تشتريه, لا, بأس]","[O, O, O, O, O, O, O, O, O, O, O, B-positive, I-positive, I-positive, I-positive, I-positive, I-positive, I-positive, I-positive, O, B-positive, I-positive, I-positive, I-positive, I-positive, I-positive, I-positive, I-positive, I-positive, I-positive, O, B-neutral, I-neutral, I-neutral, I-neutral, I-neutral, O, B-neutral, I-neutral, I-neutral, I-neutral, I-neutral, O, O, B-positive, I-positive, I-positive, B-positive, I-positive, I-positive, I-positive, I-positive, I-positive, I-positive]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 3, 3, 3, 3, 3, 0, 3, 3, 3, 3, 3, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]"
2,في سابقة عربية هي الأولى أعلن المغرب إدراج التاريخ والثقافة اليهودية في مناهج التدريس عبر إجراء إصلاح تربوي شامل يأخذ بعين الاعتبار خصوصيات الجالية اليهودية في المغرب,"[في, سابقة, عربية, هي, الأولى, أعلن, المغرب, إدراج, التاريخ, والثقافة, اليهودية, في, مناهج, التدريس, عبر, إجراء, إصلاح, تربوي, شامل, يأخذ, بعين, الاعتبار, خصوصيات, الجالية, اليهودية, في, المغرب]","[O, O, O, O, O, O, O, B-neutral, I-neutral, I-neutral, I-neutral, I-neutral, I-neutral, I-neutral, O, O, B-positive, I-positive, I-positive, O, O, O, B-neutral, I-neutral, I-neutral, I-neutral, I-neutral]","[0, 0, 0, 0, 0, 0, 0, 3, 3, 3, 3, 3, 3, 3, 0, 0, 1, 1, 1, 0, 0, 0, 3, 3, 3, 3, 3]"
3,ﺃﺟﺎﺏ ﺍلتلميذ ﺑﺘﻠﻌﺜﻢ ﻭ ﻋﻠﻰ ﺍﻟﻔﻮﺭ ﺃثنت ﻋﻠﻴﻪ ﺍلمعلمة ﺛﻨﺎﺀً ﻋﻄﺮﺍً ﻭ ﺃﻣﺮت ﺍلتلاميذ ﺑﺎﻟﺘﺼﻔﻴﻖ ﻟﻪ ﺍلتلاميذ ﺑﻴﻦ ﻣﺬﻫﻮﻝ ﻭ ﻣﺸﺪﻭﻩ و ﻣﺘﻌﺠﺐ ﻭ ﻣﺴﺘﻐﺮﺏ ﺗﻜﺮﺭ ﺍﻟﻤﺸﻬﺪ ﺧﻼﻝ ﺃﺳﺒﻮﻉ ﺑﺄﺳﺎﻟﻴﺐ,"[ﺃﺟﺎﺏ, ﺍلتلميذ, ﺑﺘﻠﻌﺜﻢ, ﻭ, ﻋﻠﻰ, ﺍﻟﻔﻮﺭ, ﺃثنت, ﻋﻠﻴﻪ, ﺍلمعلمة, ﺛﻨﺎﺀ, ً, ﻋﻄﺮﺍ, ً, ﻭ, ﺃﻣﺮت, ﺍلتلاميذ, ﺑﺎﻟﺘﺼﻔﻴﻖ, ﻟﻪ, ﺍلتلاميذ, ﺑﻴﻦ, ﻣﺬﻫﻮﻝ, ﻭ, ﻣﺸﺪﻭﻩ, و, ﻣﺘﻌﺠﺐ, ﻭ, ﻣﺴﺘﻐﺮﺏ, ﺗﻜﺮﺭ, ﺍﻟﻤﺸﻬﺪ, ﺧﻼﻝ, ﺃﺳﺒﻮﻉ, ﺑﺄﺳﺎﻟﻴﺐ]","[B-neutral, I-neutral, O, O, O, O, B-positive, I-positive, I-positive, I-positive, I-positive, I-positive, I-positive, O, B-positive, I-positive, I-positive, I-positive, B-positive, I-positive, I-positive, I-positive, I-positive, I-positive, I-positive, I-positive, I-positive, I-positive, O, O, O, O]","[3, 3, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0]"
4,وزارة التعليم في المغرب مع أسا

## Preprocessing the data

Before we can feed those texts to our model, we need to preprocess them. This is done by a  Transformers `Tokenizer` which will (as the name indicates) tokenize the inputs (including converting the tokens to their corresponding IDs in the pretrained vocabulary) and put it in a format the model expects, as well as generate the other inputs that model requires.

To do all of this, we instantiate our tokenizer with the `AutoTokenizer.from_pretrained` method, which will ensure:

- we get a tokenizer that corresponds to the model architecture we want to use,
- we download the vocabulary used when pretraining this specific checkpoint.

That vocabulary will be cached, so it's not downloaded again the next time we run the cell.

In [30]:
!pip install sentencepiece

In [31]:


from transformers import AutoTokenizer, AutoModel
#load AEBERT model from huggingface
ARBERT_tokenizer = AutoTokenizer.from_pretrained("UBC-NLP/ARBERTv2")
ARBERT_model = AutoModel.from_pretrained("UBC-NLP/ARBERTv2")
#CAMeL-Lab/bert-base-arabic-camelbert-msa
#aubmindlab/bert-base-arabertv2
#ARBERT_tokenizer = AutoTokenizer.from_pretrained("CAMeL-Lab/bert-base-arabic-camelbert-msa")
#ARBERT_model = AutoModel.from_pretrained("CAMeL-Lab/bert-base-arabic-camelbert-msa")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: UBC-NLP/ARBERTv2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


The following assertion ensures that our tokenizer is a fast tokenizers (backed by Rust) from the  Tokenizers library. Those fast tokenizers are available for almost all models, and we will need some of the special features they have for our preprocessing.

In [32]:
import transformers
assert isinstance(ARBERT_tokenizer, transformers.PreTrainedTokenizerFast)

Note that transformers are often pretrained with subword tokenizers, meaning that even if your inputs have been split into words already, each of those words could be split again by the tokenizer. Let's look at an example of that:

In [33]:
example = datasets["train"][4]
print(example)
print(example["tokens"])

{'sentence': 'بعيدا عن سياسة التطبيل و المضلومية ، اليكم أزمة المغرب الحقيقية خلط المفاهيم و سوء التربية غياب الأسرة فمواكبة العصر و حماية الأبناء تدهور التعليم و توغل الدعششة في المجتمع المغربي اي واحد لام البنت على خروجها أو لبسها المفروض يراجع نفسو لأنه مهما كانت لايحق لأحد لمسها', 'tokens': ['بعيدا', 'عن', 'سياسة', 'التطبيل', 'و', 'المضلومية', '،', 'اليكم', 'أزمة', 'المغرب', 'الحقيقية', 'خلط', 'المفاهيم', 'و', 'سوء', 'التربية', 'غياب', 'الأسرة', 'فمواكبة', 'العصر', 'و', 'حماية', 'الأبناء', 'تدهور', 'التعليم', 'و', 'توغل', 'الدعششة', 'في', 'المجتمع', 'المغربي', 'اي', 'واحد', 'لام', 'البنت', 'على', 'خروجها', 'أو', 'لبسها', 'المفروض', 'يراجع', 'نفسو', 'لأنه', 'مهما', 'كانت', 'لايحق', 'لأحد', 'لمسها'], 'ner_tags': ['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-negative', 'I-negative', 'I-negative', 'B-negative', 'I-negative', 'O', 'O', 'O', 'O', 'B-negative', 'I-negative', 'I-negative', 'I-negative', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O',

In [34]:
tokenized_input = ARBERT_tokenizer(example["tokens"], is_split_into_words=True)
tokens = ARBERT_tokenizer.convert_ids_to_tokens(tokenized_input["input_ids"])
print(tokens)

['[CLS]', 'بعيدا', 'عن', 'سياسة', 'التطب', '##يل', 'و', 'المضل', '##ومية', '،', 'اليكم', 'ازمة', 'المغرب', 'الحقيقية', 'خلط', 'المفاهيم', 'و', 'سوء', 'التربية', 'غياب', 'الاسرة', 'فمو', '##اك', '##بة', 'العصر', 'و', 'حماية', 'الابناء', 'تدهور', 'التعليم', 'و', 'توغل', 'الدع', '##شش', '##ة', 'في', 'المجتمع', 'المغربي', 'اي', 'واحد', 'لام', 'البنت', 'على', 'خروجها', 'او', 'لبس', '##ها', 'المفروض', 'يراجع', 'نفس', '##و', 'لانه', 'مهما', 'كانت', 'لاي', '##حق', 'لاحد', 'لمسها', '[SEP]']




This means that we need to do some processing on our labels as the input ids returned by the tokenizer are longer than the lists of labels our dataset contain, first because some special tokens might be added (we can a `[CLS]` and a `[SEP]` above) and then because of those possible splits of words in multiple tokens:

In [35]:
len(example[f"{task}_tags"]), len(tokenized_input["input_ids"])

(48, 59)

Thankfully, the tokenizer returns outputs that have a `word_ids` method which can help us.

In [36]:
print(tokenized_input.word_ids())

[None, 0, 1, 2, 3, 3, 4, 5, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 18, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 27, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 38, 39, 40, 41, 41, 42, 43, 44, 45, 45, 46, 47, None]


As we can see, it returns a list with the same number of elements as our processed input ids, mapping special tokens to `None` and all other tokens to their respective word. This way, we can align the labels with the processed input ids.

In [37]:
word_ids = tokenized_input.word_ids()
aligned_labels = [-100 if i is None else example[f"{task}_tags"][i] for i in word_ids]
print(len(aligned_labels), len(tokenized_input["input_ids"]))

59 59


Here we set the labels of all special tokens to -100 (the index that is ignored by PyTorch) and the labels of all other tokens to the label of the word they come from. Another strategy is to set the label only on the first token obtained from a given word, and give a label of -100 to the other subtokens from the same word. We propose the two strategies here, just change the value of the following flag:

In [38]:
label_all_tokens = True

We're now ready to write the function that will preprocess our samples. We feed them to the `tokenizer` with the argument `truncation=True` (to truncate texts that are bigger than the maximum size allowed by the model) and `is_split_into_words=True` (as seen above). Then we align the labels with the token ids using the strategy we picked:

In [39]:
def tokenize_and_align_labels(examples):
    tokenized_inputs = ARBERT_tokenizer(examples["tokens"], truncation=True, is_split_into_words=True)

    labels = []
    for i, label in enumerate(examples[f"{task}_tags1"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            # Special tokens have a word id that is None. We set the label to -100 so they are automatically
            # ignored in the loss function.
            if word_idx is None:
                label_ids.append(-100)
            # We set the label for the first token of each word.
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            # For the other tokens in a word, we set the label to either the current label or -100, depending on
            # the label_all_tokens flag.
            else:
                label_ids.append(label[word_idx] if label_all_tokens else -100)
            previous_word_idx = word_idx

        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

This function works with one or several examples. In the case of several examples, the tokenizer will return a list of lists for each key:

In [40]:
tokenize_and_align_labels(datasets['train'][:5])

{'input_ids': [[2, 1689, 4933, 2785, 1631, 4025, 213, 1876, 1840, 15798, 213, 46972, 13851, 40871, 37677, 12801, 1633, 20766, 43497, 213, 2303, 2156, 5477, 3287, 91258, 26832, 3575, 2944, 213, 2277, 3461, 92575, 1631, 33471, 3283, 1699, 68797, 1039, 1631, 2785, 47386, 1720, 10409, 1633, 10207, 16057, 213, 13392, 2256, 1968, 7153, 4025, 7341, 3], [2, 6537, 17893, 1663, 3348, 14238, 9449, 213, 10872, 6704, 2026, 213, 7677, 4210, 11952, 8681, 12223, 6803, 9102, 3522, 37952, 1631, 6305, 2134, 1748, 5873, 2597, 2477, 5362, 2785, 9849, 17395, 1631, 1967, 2579, 245, 5192, 19661, 2477, 8263, 2785, 9849, 27076, 2026, 2907, 3], [2, 2461, 2785, 1650, 4973, 9774, 4839, 4200, 35076, 10976, 2333, 3011, 5262, 213, 1924, 5697, 15594, 4280, 9449, 2053, 3443, 1671, 38542, 4280, 17648, 214, 13149, 1778, 1863, 23638, 19023, 9449, 2679, 7709, 4095, 1744, 9806, 1663, 27962, 7112, 1633, 23828, 2914, 4975, 8748, 1631, 3095, 3], [2, 4472, 1671, 4618, 5110, 2467, 34797, 2953, 1728, 1010, 10323, 11948, 1631, 402

To apply this function on all the sentences (or pairs of sentences) in our dataset, we just use the `map` method of our `dataset` object we created earlier. This will apply the function on all the elements of all the splits in `dataset`, so our training, validation and testing data will be preprocessed in one single command.

In [41]:
tokenized_datasets = datasets.map(tokenize_and_align_labels, batched=True)

Map:   0%|          | 0/952 [00:00<?, ? examples/s]

Map:   0%|          | 0/136 [00:00<?, ? examples/s]

Map:   0%|          | 0/272 [00:00<?, ? examples/s]

Even better, the results are automatically cached by the  Datasets library to avoid spending time on this step the next time you run your notebook. The Datasets library is normally smart enough to detect when the function you pass to map has changed (and thus requires to not use the cache data). For instance, it will properly detect if you change the task in the first cell and rerun the notebook.  Datasets warns you when it uses cached files, you can pass `load_from_cache_file=False` in the call to `map` to not use the cached files and force the preprocessing to be applied again.

Note that we passed `batched=True` to encode the texts by batches together. This is to leverage the full benefit of the fast tokenizer we loaded earlier, which will use multB-threading to treat the texts in a batch concurrently.

## Fine-tuning the model

Now that our data is ready, we can download the pretrained model and fine-tune it. Since all our tasks are about token classification, we use the `AutoModelForTokenClassification` class. Like with the tokenizer, the `from_pretrained` method will download and cache the model for us. The only thing we have to specify is the number of labels for our problem (which we can get from the features, as seen before):

In [42]:
from transformers import AutoModelForTokenClassification, TrainingArguments, Trainer
model = AutoModelForTokenClassification.from_pretrained("UBC-NLP/ARBERTv2", num_labels=5)


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: UBC-NLP/ARBERTv2
Key                                        | Status     | 
-------------------------------------------+------------+-
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identic

The warning is telling us we are throwing away some weights (the `vocab_transform` and `vocab_layer_norm` layers) and randomly initializing some other (the `pre_classifier` and `classifier` layers). This is absolutely normal in this case, because we are removing the head used to pretrain the model on a masked language modeling objective and replacing it with a new head for which we don't have pretrained weights, so the library warns us we should fine-tune this model before using it for inference, which is exactly what we are going to do.

To instantiate a `Trainer`, we will need to define three more things. The most important is the [`TrainingArguments`](https://huggingface.co/transformers/main_classes/trainer.html#transformers.TrainingArguments), which is a class that contains all the attributes to customize the training. It requires one folder name, which will be used to save the checkpoints of the model, and all other arguments are optional:

In [43]:
args = TrainingArguments(
    f"test-{task}",
    eval_strategy="epoch",
    save_strategy="epoch",  # Added to match evaluation frequency
    learning_rate=2e-5,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    num_train_epochs=6,
    logging_dir='./logs',
    logging_steps=2,
    weight_decay=0.001,
    # load_best_model_at_end=True,
    # metric_for_best_model="accuracy",  # Required if load_best_model_at_end=True
    # greater_is_better=True,  # Required if load_best_model_at_end=True
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Here we set the evaluation to be done at the end of each epoch, tweak the learning rate, use the `batch_size` defined at the top of the notebook and customize the number of epochs for training, as well as the weight decay.

Then we will need a data collator that will batch our processed examples together while applying padding to make them all the same size (each pad will be padded to the length of its longest example). There is a data collator for this task in the Transformers library, that not only pads the inputs, but also the labels:

In [44]:
from transformers import DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(ARBERT_tokenizer)

The last thing to define for our `Trainer` is how to compute the metrics from the predictions. Here we will load the [`seqeval`](https://github.com/chakkB-works/seqeval) metric (which is commonly used to evaluate results on the CONLL dataset) via the Datasets library.

In [45]:
#metric = load_metric("seqeval")
metric = evaluate.load("seqeval")

This metric takes list of labels for the predictions and references:

In [46]:
labels = [label_list[i] for i in example[f"{task}_tags1"]]
metric.compute(predictions=[labels], references=[labels])

{'negative': {'precision': np.float64(1.0),
  'recall': np.float64(1.0),
  'f1': np.float64(1.0),
  'number': np.int64(9)},
 'positive': {'precision': np.float64(1.0),
  'recall': np.float64(1.0),
  'f1': np.float64(1.0),
  'number': np.int64(39)},
 'overall_precision': np.float64(1.0),
 'overall_recall': np.float64(1.0),
 'overall_f1': np.float64(1.0),
 'overall_accuracy': 1.0}

So we will need to do a bit of post-processing on our predictions:
- select the predicted index (with the maximum logit) for each token
- convert it to its string label
- ignore everywhere we set a label of -100

The following function does all this post-processing on the result of `Trainer.evaluate` (which is a namedtuple containing predictions and labels) before applying the metric:

In [47]:
import numpy as np

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    # Remove ignored index (special tokens)
    true_predictions = [
        [label_list[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [label_list[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = metric.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

Note that we drop the precision/recall/f1 computed for each category and only focus on the overall precision/recall/f1/accuracy.

Then we just need to pass all of this along with our datasets to the `Trainer`:

In [48]:
trainer = Trainer(
    model,
    args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,

    compute_metrics=compute_metrics

)

We can now finetune our model by just calling the `train` method:

In [49]:
trainer.train()

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.902867,1.055725,0.578147,0.604452,0.591007,0.568221
2,0.765657,0.995577,0.575744,0.638699,0.605590,0.583381
3,0.624357,1.010207,0.598601,0.628180,0.613034,0.593936
4,0.551817,1.046523,0.596549,0.617417,0.606804,0.591249
5,0.414750,1.091563,0.594087,0.619374,0.606467,0.592401
6,0.353644,1.112477,0.602840,0.623043,0.612775,0.598350


/usr/local/lib/python3.12/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=360, training_loss=0.6828603898485501, metrics={'train_runtime': 248.2391, 'train_samples_per_second': 23.01, 'train_steps_per_second': 1.45, 'total_flos': 179883199248480.0, 'train_loss': 0.6828603898485501, 'epoch': 6.0})